In [66]:
# Check if autoreload is loaded and load/reload accordingly
try:
    %reload_ext autoreload
except:
    %load_ext autoreload
%autoreload 2
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [67]:
forward_rates = ['k_{01}', 'k_{12}', 'k_{23}', 'k_{30}']
backward_rates = ['k_{10}', 'k_{21}', 'k_{32}']

config_params = yaml.safe_load(Path("../test_files/random_ring_models/base_4ring_rate_params.yaml").open())
rate_params = config_params['Rates']

for rate_name in forward_rates:
    rate_dict = next(item for item in rate_params if item["name"] == rate_name)
    base_rate = np.random.uniform(.5, 5)
    mut_rate = np.random.uniform(.1, base_rate) # Mutated rate must be slower than base rate
    rate_dict['rate_vals'] = [base_rate, mut_rate]
    rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

for rate_name in backward_rates:
    rate_dict = next(item for item in rate_params if item["name"] == rate_name)
    # We might not need to enforce this
    reciprical_rate = next(item for item in rate_params if item["state_list"] == rate_dict["state_list"][::-1])
    base_rate = np.random.uniform(0.01, reciprical_rate['rate_vals'][0])  # Backward rate must be slower than forward rate
    # base_rate = np.random.uniform(0.01, 1.)  # Backward rate must be slower than forward rate
    mut_rate = base_rate + np.random.uniform(0.1, 3.0) # Mutated rate must be faster than base rate
    rate_dict['rate_vals'] = [base_rate, mut_rate]
    rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

print(config_params)
yaml.safe_dump(config_params, Path("../test_files/random_ring_models/generated_4ring_rate_params.yaml").open('w'))

{'Title': 'test_4ring', 'Input': {'seq_length': 7, 'template': 'AAAAAAA', 'values': ['A', 'B']}, 'States': ['0', '1', '2', '3'], 'Rates': [{'name': 'k_{01}', 'state_list': ['0', '1'], 'input_range': [0, 1], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([3.1108605096687496, 2.434116406804906]))', 'rate_vals': [3.1108605096687496, 2.434116406804906]}, {'name': 'k_{10}', 'state_list': ['1', '0'], 'input_range': [1, 2], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([1.3635393517254886, 1.6051408887683134]))', 'rate_vals': [1.3635393517254886, 1.6051408887683134]}, {'name': 'k_{12}', 'state_list': ['1', '2'], 'input_range': [2, 3], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([3.762069157449469, 2.8203881095832566]))', 'rate_vals': [3.762069157449469, 2.8203881095832566]}, {'name': 'k_{21}', 'state_list': ['2', '1'], 'input_range': [3, 4], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rat

In [ ]:
from elektrum.king_altman_kinetic_model import  KingAltmanKineticModel
# yaml_file = "/Users/alamson/projects/Elektrum/test_files/test_4ring/test_4ring_rate_params.yaml"
yaml_file = "/Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml"
print(f"Loading model from {yaml_file}")
# model = ModernKineticModel(yaml_file)
model = KingAltmanKineticModel(yaml_file)
print(model.lab_enc.transform(list(model.template)))
print(model.template)

# Use the model
# if model.config.template:
activity = model.get_activity(model.template)
print(f"Template activity: {activity}")

model

Loading model from /Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml


AttributeError: 'str' object has no attribute 'tolist'